# 🎬 YouTube Shorts ワンクリック自動生成

---

## ✏️ 毎回やること（セル1だけ変える）

| 設定項目 | 説明 |
|---|---|
| `YOUTUBE_API_KEY` | **初回だけ**入力。Google Cloud Console で取得（**無料**） |
| `CLAUDE_API_KEY` | **初回だけ**入力。console.anthropic.com で取得（有料・`sk-ant-`で始まる） |
| `PEXELS_API_KEY` | **初回だけ**入力。pexels.com/api で取得（**無料**） |
| `THEME` | **毎回**テーマを書き換える |
| `VIDEO_COUNT` | 生成する本数（1〜10） |

あとは「**ランタイム → すべてのセルを実行**」をクリックするだけ！

---

## 💰 Claude APIの費用目安
| 使い方 | 費用 |
|---|---|
| 1回10本生成 | 約5円 |
| 毎日10本 × 30日 | 約160円/月 |

---

## 🎨 画像スタイル
| スタイル名 | 見た目 | 向いているテーマ |
|---|---|---|
| `realistic` | 写真そのまま | 筋トレ・料理・ビジネス |
| `anime` | アニメ風 | 恋愛・感情・エンタメ |
| `manga` | 漫画風（白黒） | 怖い話・歴史・雑学 |
| `illustration` | イラスト風 | 子ども向け・ライフスタイル |

---

## ⏱ 目安時間
| 本数 | 目安 |
|---|---|
| 1本 | 約5〜10分 |
| 5本 | 約25〜40分 |
| 10本 | 約50〜80分 |

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  セル1: テーマと本数だけ変える（APIキーは初回のみ）  ║
# ╚══════════════════════════════════════════════════════╝

# ── ① APIキー（初回のみ。2回目以降は自動読み込み）──────
# 空白のままでOK → Colabシークレットから自動取得します
YOUTUBE_API_KEY = ''
CLAUDE_API_KEY  = ''
PEXELS_API_KEY  = ''

# ── ② テーマ（毎回変える） ──────────────────────────────
THEME = 'ダイエット'
#  例: 'ダイエット' / '筋トレ' / 'NISA' / '投資' / '英語学習' / '副業'

# ── ③ 生成する動画の本数 ────────────────────────────────
VIDEO_COUNT = 10

# ── ④ 動画の長さ ─────────────────────────────────────────
DURATION = 45

# ── ⑤ 画像スタイル ───────────────────────────────────────
IMAGE_STYLE = 'realistic'
# IMAGE_STYLE = 'anime'
# IMAGE_STYLE = 'manga'
# IMAGE_STYLE = 'illustration'

# ── ⑥ ループ再生最適化 ──────────────────────────────────────
# True: 最後のシーンを最初と同じカット＋「もう一度見てみて！」にする
#       YouTubeがループ再生するときに視覚的に自然につながる
LOOP_STRUCTURE = True

# ╔══════════════════════════════════════════════════════╗
# ║  ↑ 変えるのはここまで。以下は触らなくてOK            ║
# ╚══════════════════════════════════════════════════════╝

# ── APIキーをColabシークレットから自動取得 ───────────────
# 【初回設定】左サイドバーの 🔑 鍵アイコン → 以下3つを登録:
#   YOUTUBE_API_KEY / CLAUDE_API_KEY / PEXELS_API_KEY
#   「ノートブックからのアクセスを許可」をONにする
try:
    from google.colab import userdata as _ud
    def _clean_key(v): return ''.join(c for c in (v or '') if 0x21 <= ord(c) <= 0x7E)
    if not YOUTUBE_API_KEY.strip():
        YOUTUBE_API_KEY = _clean_key(_ud.get('YOUTUBE_API_KEY'))
    if not CLAUDE_API_KEY.strip():
        CLAUDE_API_KEY  = _clean_key(_ud.get('CLAUDE_API_KEY'))
    if not PEXELS_API_KEY.strip():
        PEXELS_API_KEY  = _clean_key(_ud.get('PEXELS_API_KEY'))
    _src = 'Colabシークレット'
except Exception:
    _src = 'セル内の直接入力'

# ── バリデーション ────────────────────────────────────────
if not YOUTUBE_API_KEY.strip():
    raise ValueError(
        '❌ YOUTUBE_API_KEY が見つかりません。\n'
        '   左サイドバーの 🔑 → 「YOUTUBE_API_KEY」を登録してください。'
    )
if not CLAUDE_API_KEY.strip():
    raise ValueError(
        '❌ CLAUDE_API_KEY が見つかりません。\n'
        '   左サイドバーの 🔑 → 「CLAUDE_API_KEY」を登録してください。'
    )
if not PEXELS_API_KEY.strip():
    raise ValueError(
        '❌ PEXELS_API_KEY が見つかりません。\n'
        '   左サイドバーの 🔑 → 「PEXELS_API_KEY」を登録してください。'
    )

import os
os.makedirs('/content/output', exist_ok=True)

print(f'🔑 APIキー読み込み元: {_src}')
print(f'  テーマ      : {THEME}')
print(f'  生成本数    : {VIDEO_COUNT}本')
print(f'  目標秒数    : {DURATION}秒（自動調整）')
print(f'  画像スタイル: {IMAGE_STYLE}')
print()
print('✅ 設定完了！')


In [ ]:
# 【自動】必要なツールをインストール（触らなくてOK）
!pip install -q gtts requests opencv-python-headless
!apt-get install -q -y ffmpeg fonts-noto-cjk p7zip-full
print('✅ ツールのインストール完了')

# ── VOICEVOX Engine セットアップ（完全無料・ローカル動作）──
import subprocess as _sp, os as _os, time as _time
import requests as _req
VV_PORT = 50021
VV_URL  = f'http://127.0.0.1:{VV_PORT}'
VV_DIR  = '/content/voicevox_engine'
VV_SPEAKER = 3  # ずんだもん（ノーマル）
VV_AVAILABLE = False

if not _os.path.exists(f'{VV_DIR}/run'):
    print('🔊 VOICEVOXエンジンをダウンロード中（初回のみ・数分かかります）...')
    try:
        _rel = _req.get(
            'https://api.github.com/repos/VOICEVOX/voicevox_engine/releases/latest',
            timeout=30).json()
        # v0.20以降は .7z.001 形式（Linux x64 CPU版）
        _asset = next(
            (a for a in _rel['assets']
             if 'linux-cpu-x64' in a['name'] and a['name'].endswith('.7z.001')),
            None
        )
        if _asset is None:
            raise RuntimeError(f'Linux CPU x64アセットが見つかりません。利用可能: {[a["name"] for a in _rel["assets"]]}')
        _fn = f'/tmp/{_asset["name"]}'
        print(f'  ダウンロード: {_asset["name"]}')
        _sp.run(['wget', '-q', '--show-progress', '-O', _fn,
                 _asset['browser_download_url']], check=True)
        _os.makedirs(VV_DIR, exist_ok=True)
        _sp.run(['7z', 'x', _fn, f'-o{VV_DIR}', '-y'],
                check=True, capture_output=True)
        _os.remove(_fn)
        # 7z展開でサブディレクトリが作られた場合は1段上げる
        _subdirs = [d for d in _os.listdir(VV_DIR)
                    if _os.path.isdir(f'{VV_DIR}/{d}') and not d.startswith('.')]
        if _subdirs and not _os.path.exists(f'{VV_DIR}/run'):
            _sub = f'{VV_DIR}/{_subdirs[0]}'
            for _item in _os.listdir(_sub):
                _sp.run(['mv', f'{_sub}/{_item}', VV_DIR], check=True)
            _os.rmdir(_sub)
        _sp.run(['chmod', '+x', f'{VV_DIR}/run'], check=False)
        print('✅ VOICEVOXダウンロード・展開完了')
    except Exception as _e:
        print(f'⚠️ VOICEVOXダウンロード失敗（gTTSで代替します）: {_e}')

if _os.path.exists(f'{VV_DIR}/run'):
    _sp.Popen(
        [f'{VV_DIR}/run', '--host', '127.0.0.1', '--port', str(VV_PORT)],
        stdout=_sp.DEVNULL, stderr=_sp.DEVNULL
    )
    print('🔊 VOICEVOXエンジン起動中（最大60秒）...')
    for _i in range(60):
        _time.sleep(1)
        try:
            if _req.get(f'{VV_URL}/version', timeout=2).status_code == 200:
                VV_AVAILABLE = True
                print(f'✅ VOICEVOX起動完了（{_i+1}秒）→ 高品質音声モード')
                break
        except:
            pass
    if not VV_AVAILABLE:
        print('⚠️ VOICEVOX起動タイムアウト → gTTSにフォールバック')
else:
    print('⚠️ VOICEVOXが見つかりません → gTTSで音声生成します')


In [ ]:
# 【切断防止】Colabの自動切断を防ぐ（動画生成中に実行しておく）
# ※ このセルを実行してからセル4（動画生成）を実行してください
import threading, time

_keepalive_running = True

def _keepalive_loop():
    """90秒ごとにColabのアイドル検知をリセット"""
    import IPython
    count = 0
    while _keepalive_running:
        time.sleep(90)
        count += 1
        IPython.display.display(IPython.display.Javascript(
            'google.colab.kernel.proxyPort(0)'
        ))
        print(f'\r⏳ 切断防止 keepalive #{count} ({count*90//60}分経過)', end='', flush=True)

_t = threading.Thread(target=_keepalive_loop, daemon=True)
_t.start()
print('✅ 切断防止スタート（動画生成が終わったら気にしなくてOK）')


In [ ]:
# 【自動】① キーワード調査 → ② Shortsトレンド → ③ 長尺・最新情報 → ④ AI統合分析 → ⑤ 切り口生成

import requests as _req, json as _json, re, datetime

# ── 情報取得関数 ─────────────────────────────────────────────

def fetch_search_keywords(theme):
    """ユーチューブオートコンプリートで「今まさに検索されているワード」を取得（APIキー不要）"""
    try:
        r = _req.get(
            'https://suggestqueries.google.com/complete/search',
            params={'client': 'youtube', 'hl': 'ja', 'ds': 'yt', 'q': theme},
            timeout=10
        )
        if r.status_code != 200:
            return []
        text = r.text
        if text.startswith('window.'):
            text = text[text.index('(') + 1:text.rindex(')')]
        data = _json.loads(text)
        return [item[0] for item in data[1] if isinstance(item, (list, tuple)) and item][:12]
    except Exception as e:
        print(f'  ⚠ キーワード取得失敗: {e}')
        return []

def fetch_youtube_shorts_trends(theme):
    """2パス検索: ①直近90日トレンド(relevance) + ②全期間人気(viewCount) → 重複排除"""
    after = (datetime.datetime.utcnow() - datetime.timedelta(days=90)).strftime('%Y-%m-%dT%H:%M:%SZ')
    base = {
        'part': 'snippet', 'q': f'{theme} shorts', 'type': 'video',
        'regionCode': 'JP', 'relevanceLanguage': 'ja',
        'key': YOUTUBE_API_KEY, 'videoDuration': 'short',
    }
    passes = [
        {**base, 'order': 'relevance', 'maxResults': 15, 'publishedAfter': after},
        {**base, 'order': 'viewCount',  'maxResults': 10},
    ]
    results, seen = [], set()
    for params in passes:
        try:
            r = _req.get('https://www.googleapis.com/youtube/v3/search', params=params, timeout=15)
            if r.status_code != 200:
                print(f'  ⚠ Shorts検索API: {r.status_code}')
                continue
            for item in r.json().get('items', []):
                vid = item['id'].get('videoId', '')
                if vid and vid not in seen:
                    seen.add(vid)
                    results.append({
                        'id':      vid,
                        'title':   item['snippet'].get('title', ''),
                        'channel': item['snippet'].get('channelTitle', ''),
                    })
        except Exception as e:
            print(f'  ⚠ YouTube API失敗: {e}')
    return results

def fetch_longform_content(theme):
    """長尺動画（4～20分）から最新エビデンス・新成分・トレンド知識を収集"""
    after = (datetime.datetime.utcnow() - datetime.timedelta(days=180)).strftime('%Y-%m-%dT%H:%M:%SZ')
    queries = [
        f'{theme} 最新 効果 エビデンス',
        f'{theme} 方法 やり方 解説',
    ]
    results, seen = [], set()
    for q in queries:
        try:
            r = _req.get(
                'https://www.googleapis.com/youtube/v3/search',
                params={
                    'part': 'snippet', 'q': q, 'type': 'video',
                    'order': 'relevance', 'regionCode': 'JP',
                    'relevanceLanguage': 'ja', 'maxResults': 8,
                    'key': YOUTUBE_API_KEY, 'videoDuration': 'medium',
                    'publishedAfter': after,
                },
                timeout=15
            )
            if r.status_code == 200:
                for item in r.json().get('items', []):
                    vid = item['id'].get('videoId', '')
                    title = item['snippet'].get('title', '')
                    if vid and vid not in seen and title:
                        seen.add(vid)
                        results.append(title)
        except Exception as e:
            print(f'  ⚠ 長尺動画取得失敗: {e}')
    return results[:15]

def fetch_video_stats(video_ids):
    """再生数・いいね・コメ・タグ・尺を取得。60秒超えはShortsではないので除外。"""
    if not video_ids: return []
    def _parse_dur(iso):
        m = re.search(r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?', iso or '')
        if not m: return 999
        h, mn, s = (int(x or 0) for x in m.groups())
        return h * 3600 + mn * 60 + s
    try:
        r = _req.get(
            'https://www.googleapis.com/youtube/v3/videos',
            params={'part': 'snippet,statistics,contentDetails',
                    'id': ','.join(video_ids[:20]), 'key': YOUTUBE_API_KEY},
            timeout=15
        )
        if r.status_code != 200:
            print(f'  ⚠ YouTube統計API: {r.status_code}')
            return []
        results = []
        for item in r.json().get('items', []):
            snip  = item.get('snippet', {})
            stats = item.get('statistics', {})
            dur   = _parse_dur(item.get('contentDetails', {}).get('duration', ''))
            if dur > 60: continue
            results.append({
                'title':       snip.get('title', ''),
                'description': snip.get('description', '')[:200],
                'tags':        snip.get('tags', [])[:8],
                'duration_s':  dur,
                'views':       int(stats.get('viewCount',   0)),
                'likes':       int(stats.get('likeCount',   0)),
                'comments':    int(stats.get('commentCount',0)),
            })
        return sorted(results, key=lambda x: x['views'], reverse=True)
    except Exception as e:
        print(f'  ⚠ 統計取得失敗: {e}')
        return []

CLAUDE_URL = 'https://api.anthropic.com/v1/messages'

def call_ai(prompt, tokens=4096):
    res = _req.post(
        CLAUDE_URL,
        headers={'x-api-key': CLAUDE_API_KEY,
                 'anthropic-version': '2023-06-01',
                 'content-type': 'application/json'},
        json={'model': 'claude-haiku-4-5-20251001',
              'max_tokens': tokens,
              'messages': [{'role': 'user', 'content': prompt}]},
        timeout=120
    )
    if res.status_code != 200:
        err = res.text[:300].replace(CLAUDE_API_KEY, '***')
        raise RuntimeError(f'Claude APIエラー ({res.status_code}): {err}')
    return res.json()['content'][0]['text']

def clean_title(t):
    t = re.sub(r'^#+\s*', '', t)
    t = re.sub(r'\*+', '', t)
    t = re.sub(r'^\d+[\.)\]]\s*', '', t)
    return t.strip()

# ══════════════════════════════════════════════════════
# ① 検索キーワード調査（オートコンプリート）
# ══════════════════════════════════════════════════════
print(f'🔍 「{THEME}」の検索キーワードを調査中...')
kw_suggestions = fetch_search_keywords(THEME)
kw_text = '・'.join(kw_suggestions) if kw_suggestions else 'データなし'
print(f'  ✓ 検索ワード({len(kw_suggestions)}件): {kw_text[:80]}')

# ══════════════════════════════════════════════════════
# ② YouTube Shorts トレンド取得（直近90日 + 歴代人気）
# ══════════════════════════════════════════════════════
print('\n📺 YouTube Shortsトレンドを検索中...')
yt_basic = fetch_youtube_shorts_trends(THEME)

if yt_basic:
    print(f'  ✓ {len(yt_basic)}件の動画を発見')
    yt_stats = fetch_video_stats([v['id'] for v in yt_basic])
    if yt_stats:
        print(f'  ✓ Shorts確認済み {len(yt_stats)}本の詳細取得完了')
        print('\n  🔥 最も再生されている動画:')
        for v in yt_stats[:3]:
            print(f'    再生:{v["views"]:,} │ {v["title"][:45]}')
        analysis_lines = []
        for i, v in enumerate(yt_stats[:15]):
            tags_str = '・'.join(v['tags'][:6]) if v['tags'] else 'タグなし'
            analysis_lines.append(
                f'{i+1}. 「{v["title"]}」{v.get("duration_s","")}秒\n'
                f'   再生:{v["views"]:,} / いいね:{v["likes"]:,} / コメ:{v["comments"]:,}\n'
                f'   タグ: {tags_str}\n'
                f'   説明: {v["description"][:100]}'
            )
        yt_data_text = '\n'.join(analysis_lines)
    else:
        yt_data_text = '\n'.join(f'{i+1}. 「{v["title"]}」' for i, v in enumerate(yt_basic))
else:
    print('  ⚠ Shorts取得できず。AI知識でトレンド分析します。')
    yt_data_text = f'テーマ「{THEME}」の一般的なトレンド情報'

# ══════════════════════════════════════════════════════
# ③ 長尺動画から最新エビデンス・トレンド情報を収集
# ══════════════════════════════════════════════════════
print('\n📹 長尺動画（最新解説・エビデンス・製品情報）を調査中...')
longform_titles = fetch_longform_content(THEME)
if longform_titles:
    print(f'  ✓ {len(longform_titles)}件の長尺動画を発見')
    longform_text = '\n'.join(f'・{t}' for t in longform_titles)
else:
    print('  ⚠ 長尺動画データなし')
    longform_text = 'データなし'

# ══════════════════════════════════════════════════════
# ④ Claude AI統合分析（Shorts + 長尺 + キーワード 三点合わせ）
# ══════════════════════════════════════════════════════
print('\n🤖 Claude AIで統合分析中...')
trend = call_ai(
    f'テーマ「{THEME}」のYouTube総合リサーチデータ\n\n'
    f'【A. 今まさに検索されているキーワード（オートコンプリート実測値）】\n{kw_text}\n\n'
    f'【B. Shorts上位動画（直近90日トレンド＋歴代人気 混合・実数値）】\n{yt_data_text}\n\n'
    f'【C. 長尺動画タイトル（最新解説・エビデンス・製品・成分情報源）】\n{longform_text}\n\n'
    f'上要3つのデータを統合して、以下を日本語・箇条書きで出力してください:\n\n'
    f'　1. 今バズるタイトル公式（3～5パターン）、\n'
    f'実際のデータから抽出した構造（数字＋煽り語の組み合わせ、具体例付き）\n\n'
    f'　2. 冠頭3秒フックフレーズ（5例）\n'
    f'視聴者が手を止める具体的な日本語フレーズ\n\n'
    f'　3. 今ホットな最新ネタ（Shortsに落とし込める情報）\n'
    f'長尺動画から抽出した最新エビデンス・流行成分・注目メソッドをShorts1シーン分に要約\n\n'
    f'　4. コメント誘発ネタ\n'
    f'「それ違う」「私もやってみる」が生まれやすいテーマ・断言・比較\n\n'
    f'　5. タイトルに入れるべき検索ワード（自然に入るもの）\n'
    f'オートコンプリートから抽出した、タイトルに入れると検索ヒットするワード\n\n'
    f'　6. 音声スタイル】語尾・テンポ・間の特徴（2行）'
)
print('  ✓ 統合分析完了')
print('\n  ━━━━ 分析サマリー ━━━━')
print(f'  {trend[:400]}')
print(f'  ...')

TREND_ANALYSIS = trend

# ══════════════════════════════════════════════════════
# ⑤ VIDEO_COUNT個の切り口を生成（キーワード自然組み込み）
# ══════════════════════════════════════════════════════
print(f'\n💡 切り口を{VIDEO_COUNT}個考案中...')
angles_raw = call_ai(
    f'テーマ「{THEME}」のYouTube Shortsタイトルを{VIDEO_COUNT}個出力してください。\n\n'
    f'トレンド分析（実データ）:\n{trend[:700]}\n\n'
    f'実際に検索されているワード: {kw_text[:200]}\n\n'
    f'【絶対守ること】\n'
    f'・タイトルのみを出力（説明文・番号・記号なし）\n'
    f'・1行1タイトル\n'
    f'・数字を必ず入れる（20文字以内）\n'
    f'・煽り系ワード（知らないと損、行ってはいけない、行ってみたら人生変わる）\n'
    f'・「タイトルに入れるべき検索ワード」を自然に含める（無理に入れない）\n'
    f'・長尺動画から得た最新ネタ（エビデンス・新成分・メソッド）を優先\n'
    f'・各タイトルは全て異なる切り口にする\n\n'
    f'出力例（ダイエットの場合）:\n'
    f'プロテイン飲む本当のタイミング\n'
    f'クレアチン効果が出ない３つの理由\n'
    f'食べても太らない時間帯とは',
    tokens=1024
)
ANGLES = [clean_title(l) for l in angles_raw.strip().split('\n')
          if l.strip() and len(l.strip()) > 3][:VIDEO_COUNT]
while len(ANGLES) < VIDEO_COUNT:
    ANGLES.append(f'{THEME}の秘密 Vol.{len(ANGLES)+1}')

print(f'\n生成する{VIDEO_COUNT}本:')
for i, a in enumerate(ANGLES):
    print(f'  {i+1:2d}. {a}')
print(f'\n✅ 切り口の決定完了')


In [ ]:
# 【自動】③〜⑥ 全動画を一括生成（触らなくてOK）

import cv2, numpy as np, subprocess, tempfile, time, wave, json as _json, shutil as _shutil
import scipy.io.wavfile as wio, requests as _req, re
from pathlib import Path
from gtts import gTTS

def anime(img):
    c = cv2.bilateralFilter(cv2.bilateralFilter(img,9,250,250),9,250,250)
    g = cv2.medianBlur(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY),5)
    e = cv2.cvtColor(cv2.adaptiveThreshold(g,255,cv2.ADAPTIVE_THRESH_MEAN_C,cv2.THRESH_BINARY,9,5),cv2.COLOR_GRAY2BGR)
    r = cv2.bitwise_and(c,e)
    h = cv2.cvtColor(r,cv2.COLOR_BGR2HSV).astype(np.float32)
    h[:,:,1] = np.clip(h[:,:,1]*1.5,0,255)
    return cv2.cvtColor(h.astype(np.uint8),cv2.COLOR_HSV2BGR)

def manga(img):
    g = cv2.createCLAHE(2.0,(8,8)).apply(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY))
    e = cv2.dilate(cv2.Canny(cv2.GaussianBlur(g,(3,3),0),30,100),np.ones((2,2),np.uint8))
    _,t = cv2.threshold(g,180,255,cv2.THRESH_BINARY)
    return cv2.cvtColor(cv2.addWeighted(t,.75,cv2.bitwise_not(e),.25,0),cv2.COLOR_GRAY2BGR)

def illust(img):
    c = cv2.bilateralFilter(img,15,80,80)
    h = cv2.cvtColor(c,cv2.COLOR_BGR2HSV).astype(np.float32)
    h[:,:,1]=np.clip(h[:,:,1]*1.7,0,255); h[:,:,2]=np.clip(h[:,:,2]*1.1,0,255)
    c = cv2.cvtColor(h.astype(np.uint8),cv2.COLOR_HSV2BGR)
    e = cv2.cvtColor(cv2.adaptiveThreshold(cv2.medianBlur(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY),7),255,
        cv2.ADAPTIVE_THRESH_MEAN_C,cv2.THRESH_BINARY,11,9),cv2.COLOR_GRAY2BGR)
    return cv2.bitwise_and(c,e)

STYLES = {'anime':anime,'manga':manga,'illustration':illust}

def ff(*a):
    r = subprocess.run(['ffmpeg','-y',*[str(x) for x in a]],
                       capture_output=True,text=True,timeout=300)
    if r.returncode != 0:
        raise RuntimeError(r.stderr[-800:])

def at(s):
    return f'{int(s//3600)}:{int((s%3600)//60):02d}:{int(s%60):02d}.{int((s%1)*100):02d}'

def get_wav_dur(path):
    try:
        with wave.open(str(path),'r') as wf:
            return wf.getnframes()/wf.getframerate()
    except:
        return 3.0

def clean_text(t):
    t = re.sub(r'#\S+', '', t)
    t = re.sub(r'\[速く\]|\[ゆっくり\]|\[強調\]','',t)
    t = re.sub(r'\[間\d+\.?\d*\]','、',t)
    t = re.sub(r'（ここにセリフ）|\(ここにセリフ\)|シーン\d+[（(][^)）]*[)）]\s*[:：]','',t)
    t = re.sub(r'^#+\s*','',t); t = re.sub(r'[\*_]','',t)
    return re.sub(r'\s+',' ',t).strip()

def verify_and_fix_script(scene_texts, theme, angle):
    """Step3: 文法・内容・フォーマットを自己検証 → 問題箇所を自動修正"""
    if not scene_texts:
        return scene_texts
    scenes_str = '\n'.join(f'シーン{i+1}: {t}' for i, t in enumerate(scene_texts))
    try:
        result = call_ai(
            f'以下のYouTube Shorts台本を検証し、修正したものを出力してください。\n\n'
            f'【台本】\n{scenes_str}\n\n'
            f'【テーマ】{theme}  【タイトル】{angle}\n\n'
            f'【検証・修正ルール（この順番で全シーン確認）】\n'
            f'①語尾が「～んだ」「～なんです」「～でしょう」「～むんだよ」など\n'
            f'  ロボット調なら「～だよ！」「～してみて！」「～じゃない？」に修正\n'
            f'②英語・記号・ハッシュタグが混入していたら削除\n'
            f'③15文字を超えるセリフは意味を保ったまま短縮\n'
            f'④シーン1のフックが弱ければ「え、マジ？」「知ってた？」型に強化\n'
            f'⑤内容の流れ（駅き→共感→解決→行動）が不自然なら並び替え\n\n'
            f'【出力フォーマット（厳守）】\n'
            f'修正後のセリフのみ出力。問題がないシーンもそのまま全て出力。\n'
            f'シーン1: （修正後セリフ）\n'
            f'シーン2: （修正後セリフ）\n'
            f'（元と同じシーン数で出力すること）',
            tokens=512
        )
        fixed = []
        for line in result.strip().split('\n'):
            m = re.match(r'シーン\d+[::：]\s*(.+)', line.strip())
            if m:
                t = clean_text(m.group(1).strip())
                if t: fixed.append(t)
        if len(fixed) == len(scene_texts):
            changed = sum(1 for a, b in zip(scene_texts, fixed) if a != b)
            if changed:
                print(f'  ✏️  {changed}シーンを自動修正')
            return fixed
        return scene_texts
    except Exception as e:
        print(f'  ⚠ 検証スキップ: {e}')
        return scene_texts

def tts_text_cleanse(text):
    """TTS専用テキスト補正: gTTSが自然なイントネーションで読めるよう意味・語尾を修正"""
    # ① AI特有の語尾をgTTSが自然に読める語尾に変換
    text = re.sub(r'んだよ([！！]?)', r'よ\1', text)       # 〜んだよ！ → 〜よ！
    text = re.sub(r'んだよね', 'よね', text)
    text = re.sub(r'んだね([！！]?)', r'だよね\1', text)
    text = re.sub(r'んです([よ！ね。]?)', r'よ\1', text)    # 〜んです → 〜よ
    text = re.sub(r'なんだ([！！。]?)', r'だよ\1', text)   # 〜なんだ！ → 〜だよ！
    text = re.sub(r'なんです', 'だよ', text)
    text = re.sub(r'するんだ([！！。]?)', r'するよ\1', text)
    text = re.sub(r'できるんだ([！！。]?)', r'できるよ\1', text)
    text = re.sub(r'整うんだ([！！。]?)', r'整うよ\1', text)   # ユーザー指定例
    text = re.sub(r'変わるんだ([！！。]?)', r'変わるよ\1', text)  # ユーザー指定例
    text = re.sub(r'あるんだ([！！。]?)', r'あるよ\1', text)
    text = re.sub(r'いるんだ([！！。]?)', r'いるよ\1', text)
    # ② 読点（ポーズ）挿入: gTTSが自然な間を置けるよう感嘆詞の後に読点
    text = re.sub(r'^(え)(マジ|本当|すごい|やば)', r'\1、\2', text)  # えマジ→え、マジ
    text = re.sub(r'^(実は)([^、])', r'\1、\2', text)               # 実は→実は、
    text = re.sub(r'^(ねえ|ねっ|あのね)([^、])', r'\1、\2', text)   # ねえ→ねえ、
    text = re.sub(r'^(知ってる)(\?|？)?([^、])', r'\1？\3', text)
    # ③ 数字＋単位の間にスペース: gTTSが分けて読めるよう
    text = re.sub(
        r'([一二三四五六七八九十百千万億])(キロ|グラム|センチ|メートル|ヶ月|ヵ月|カ月|週間|日間|時間)',
        r'\1 \2', text
    )
    # ④ 重複記号を整理
    text = re.sub(r'！{2,}', '！', text)
    text = re.sub(r'、{2,}', '、', text)
    return text.strip()

def preprocess_tts(text):
    """ハッシュタグ・記号を完全除去し gTTS が読めるテキストへ"""
    text = re.sub(r'#\S+', '', text)
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'[\[\](){}@&*|\\^~`<>=+_・]', '', text)
    text = re.sub(r'[^ -~　-鿿！-｠゠-ヿ぀-ゟ]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if len(text) > 16 and '、' not in text:
        mid = len(text) // 2
        for j in range(max(0, mid-4), min(len(text)-1, mid+5)):
            if text[j] in 'はがをにでもよねてし':
                text = text[:j+1] + '、' + text[j+1:]
                break
    return text.strip() or 'つぎのシーンです'

def make_tts_voicevox(text, tmp_dir, prefix, speaker=None):
    """VOICEVOX REST APIで高品質日本語音声を生成（ずんだもんノーマル）"""
    if speaker is None: speaker = VV_SPEAKER
    try:
        import requests as _r
        # Step1: audio_query
        q = _r.post(f'{VV_URL}/audio_query',
                    params={'text': text, 'speaker': speaker}, timeout=30)
        q.raise_for_status()
        query = q.json()
        # 話速・イントネーション・無音長を調整
        query['speedScale']       = 1.05   # わずかに速め（gTTSより自然）
        query['intonationScale']  = 1.15   # イントネーション強調
        query['prePhonemeLength'] = 0.05   # 発話前の無音
        query['postPhonemeLength']= 0.10   # 発話後の無音
        # Step2: synthesis
        s = _r.post(f'{VV_URL}/synthesis',
                    params={'speaker': speaker}, json=query, timeout=60)
        s.raise_for_status()
        out = tmp_dir / f'{prefix}_vv.wav'
        out.write_bytes(s.content)
        return out
    except Exception as _e:
        print(f'  ⚠ VOICEVOX TTS失敗: {_e}')
        return None

def make_tts_audio(text, tmp_dir, prefix):
    """VOICEVOX優先、失敗時はgTTSフォールバックで音声生成"""
    text = preprocess_tts(text)
    # ── VOICEVOX優先ルート ───────────────────────────────────
    if VV_AVAILABLE:
        _vv = make_tts_voicevox(text, tmp_dir, prefix)
        if _vv and _vv.exists():
            return _vv
    # ── gTTSフォールバック ────────────────────────────────────
    sr = 44100
    # 句読点の後で分割（句読点を各フレーズに含める）
    parts = [p.strip() for p in re.split(r'(?<=[。！？、])', text) if p.strip() and len(p.strip()) >= 2]
    if not parts:
        parts = [text] if text else []
    if not parts:
        return None

    wavs = []
    for j, phrase in enumerate(parts):
        mp3 = tmp_dir/f'{prefix}_{j}.mp3'
        wav = tmp_dir/f'{prefix}_{j}.wav'
        sil = tmp_dir/f'{prefix}_{j}_s.wav'
        try:
            gTTS(text=phrase, lang='ja', slow=False).save(str(mp3))
            ff('-i',str(mp3),'-filter:a',
               'silenceremove=start_periods=1:start_silence=0.02:start_threshold=-42dB'
               ':stop_periods=-1:stop_silence=0.04:stop_threshold=-42dB,atempo=1.35',
               '-ar','44100','-ac','1',str(wav))
            wavs.append(wav)
            # 句点・感嘆符・疑問符 = 180ms、読点 = 80ms の無音
            sil_ms = 140 if phrase[-1] in '。！？' else 60
            wio.write(str(sil), sr, np.zeros(int(sr*sil_ms/1000), dtype=np.float32))
            wavs.append(sil)
        except:
            pass

    if not wavs:
        return None

    out = tmp_dir/f'{prefix}_out.wav'
    if len(wavs) == 1:
        _shutil.copy(str(wavs[0]), str(out))
    else:
        lf = tmp_dir/f'{prefix}_lf.txt'
        lf.write_text('\n'.join(f"file '{p}'" for p in wavs))
        ff('-f','concat','-safe','0','-i',str(lf),'-c','copy',str(out))
    return out

def clean_subtitle_text(text):
    """字幕クレンジング: 句読点・冒頭感嘆詞を除去"""
    text = re.sub(r'[\u3002\u3001]', '', text)
    text = re.sub(r'^(\u3048\u3063|\u3042\u3063|\u306d\u3048|\u306d\u3063|\u3046\u308f|\u304a\u3063|\u3078\u3048|\u308f\u3042|\u307e\u3042|\u306f\u3042|\u306f)[\uff01\uff1f!?]?\s*', '', text)
    return text.strip()

def make_sub_cards(text, scene_dur, max_chars=12):
    """
    文節境界優先で1行/最大12文字のカードに分割。
    優先切断: 助詞(は・が・を・に・で・も・て・よ・ね・から・まで)の直後
    次点: 感嘆符・疑問符
    最終手段: max_chars 強制ぶつ切り
    Returns: [(card_text, start_offset, end_offset), ...]
    """
    base = clean_subtitle_text(text)
    base = re.sub(r'#\S+', '', base).strip()
    if not base:
        base = text[:max_chars]
    _BUNSETSU_HIGH = set('はがをにでもてよねへ')
    _KINSOKU_HEAD  = set('！？。、…）」』')

    def _find_break(s, limit):
        """s の [2, limit+2) 範囲内で最適な切断点インデックスを返す"""
        for i in range(min(len(s) - 1, limit + 2), 2, -1):
            c_prev = s[i - 1]
            c_next = s[i] if i < len(s) else ''
            if c_prev in _BUNSETSU_HIGH and c_next not in _KINSOKU_HEAD:
                return i
        for i in range(min(len(s) - 1, limit + 2), 2, -1):
            if s[i - 1] in '！？' and (i >= len(s) or s[i] not in _KINSOKU_HEAD):
                return i
        return limit

    cards = []
    rem = base
    while rem and len(cards) < 4:
        if len(rem) <= max_chars:
            cards.append(rem)
            break
        brk = _find_break(rem, max_chars)
        chunk = rem[:brk].strip()
        if not chunk:
            chunk = rem[:max_chars]
            brk   = max_chars
        cards.append(chunk)
        rem = rem[brk:].strip()
    if not cards:
        cards = [base[:max_chars]]
    while len(cards) > 1 and scene_dur / len(cards) < 0.8:
        cards.pop()
    cards = [re.sub(r'([①-⑧])', r'{\\c&H00FFFFFF&}\1{\\c&H0000FFFF&}', c) for c in cards]
    n = len(cards)
    slot = scene_dur / n
    return [(cards[j], j * slot, (j + 1) * slot - 1.0/30) for j in range(n)]

def validate_sub_cards(cards, scene_text, scene_dur, max_chars=12):
    """字幕カード自己評価: 文字数超過/禁則文頭を検知してClaude修正を1回実行
    cards は make_sub_cards が返す (text, start_off, end_off) のタプルリスト
    """
    _KINSOKU = set('\uff01\uff1f\u3002\u3001\u2026\u300d\uff09\u300b')
    def _strip(t):
        # タプルの場合はテキスト部分のみ取り出す
        s = t[0] if isinstance(t, tuple) else t
        return re.sub(r'\{[^}]*\}', '', str(s))
    def _viol(lst):
        for item in lst:
            raw = _strip(item)
            if len(raw) > max_chars: return f'文字数超過({len(raw)}文字):{raw!r}'
            if raw and raw[0] in _KINSOKU: return f'文頭禁則:{raw!r}'
        return None
    def _rebuild(texts):
        """テキストリストをタプルリストに変換（シーン尺均等配分）"""
        n = len(texts)
        while n > 1 and scene_dur / n < 0.8: n -= 1
        texts = texts[:n]; slot = scene_dur / n
        return [(texts[j], j*slot, (j+1)*slot - 1.0/30) for j in range(n)]
    v = _viol(cards)
    if not v: return cards  # 問題なし
    print(f'    ⚠ 字幕検証NG ({v}) → Claude修正ループ')
    try:
        _raw = call_ai(
            f'以下のセリフを字幕カードに分割してください。\n'
            f'セリフ：{scene_text}\n\n'
            f'【必須ルール】\n'
            f'・1カードは10文字以内\n'
            f'・句読点・記号（！？。、…）を行頭に置かない\n'
            f'・改行で最大3カードに分割\n'
            f'・カードテキストのみ出力（説明不要）',
            tokens=80)
        _lines = [re.sub(r'\{[^}]*\}', '', l.strip()) for l in _raw.strip().split('\n') if l.strip()][:4]
        _lines = [l[:max_chars] for l in _lines if l]
        if _lines and not _viol(_lines):
            print(f'    ✓ 字幕修正完了: {_lines}')
            return _rebuild(_lines)
    except Exception as _ve:
        print(f'    ⚠ Claude修正失敗: {_ve}')
    # 最終手段: タプルからテキストを取り出して強制トリム
    _safe = [_strip(c)[:max_chars] for c in cards]
    return _rebuild(_safe)

def auto_fit_fontsize(text, base=75, min_fs=48):
    """make_sub_cardsで12文字以内に制限済みのため常にbase=75ptを返す"""
    return base


def pre_render_audit(scenes, sub_events, cap_dur, keywords, theme):
    """
    レンダリング前全仕様スキャン。違反はClaude APIで自動修正（Self-Correction Loop）。
    Returns: (sub_events_fixed, all_passed: bool)
    """
    violations = []
    Y_MIN, Y_MAX = int(H * 0.60), int(H * 0.63)

    # 1. 字幕文字数チェック (≤12文字)
    sub_over = []
    for ev in sub_events:
        m = re.search(r'pos\(\d+,\d+\)\\fs\d+\}(.+)$', ev)
        if m:
            txt = re.sub(r'\{[^}]*\}', '', m.group(1))
            if len(txt) > 12:
                sub_over.append(txt)
    if sub_over:
        violations.append(f'字幕文字数超過 {len(sub_over)}件: {sub_over[:3]}')

    # 2. Y座標チェック (H*0.60 ≤ Y ≤ H*0.63)
    y_viols = []
    for ev in sub_events:
        m = re.search(r'\\pos\(\d+,(\d+)\)', ev)
        if m:
            y = int(m.group(1))
            if not (Y_MIN <= y <= Y_MAX):
                y_viols.append(y)
    if y_viols:
        violations.append(f'Y座標違反 {len(y_viols)}件: {sorted(set(y_viols))}')

    # 3. 再生秒数チェック (15s ≤ cap_dur ≤ 25s)
    if not (15.0 <= cap_dur <= 25.0):
        violations.append(f'再生秒数範囲外: {cap_dur:.2f}s (15〜25秒)')

    # 4. ブラックリスト素材チェック
    bl_hits = [kw for kw in keywords if any(b in kw.lower() for b in _KEYWORD_BLACKLIST)]
    if bl_hits:
        violations.append(f'BL素材検知 {len(bl_hits)}件: {bl_hits}')

    if not violations:
        print('  ✅ pre-render Audit: 全4チェック通過')
        return sub_events, True

    print(f'  ⚠ pre-render Audit 違反 ({len(violations)}件) → 自動修正ループ開始')
    for v in violations:
        print(f'    - {v}')

    # Y座標違反: 直接修正（Claudeは不要）
    if y_viols:
        correct_y = str(int(H * 0.62))
        sub_events = [
            re.sub(r'(\\pos\(\d+,)\d+(\))', r'\g<1>' + correct_y + r'\g<2>', ev)
            for ev in sub_events
        ]
        print(f'  ✓ Y座標を{correct_y}に自動修正')

    # 字幕文字数超過: Claude修正ループ
    if sub_over:
        try:
            fixed_evs = []
            for ev in sub_events:
                m = re.search(r'(.*pos\(\d+,\d+\)\\fs\d+\})(.+)$', ev)
                if m:
                    prefix, txt = m.group(1), m.group(2)
                    txt_clean = re.sub(r'\{[^}]*\}', '', txt)
                    if len(txt_clean) > 12:
                        resp = call_ai(
                            f'「{txt_clean}」を10文字以内に要約してください。テキストのみ出力。',
                            tokens=30
                        ).strip()[:12]
                        fixed_evs.append(prefix + resp)
                        continue
                fixed_evs.append(ev)
            sub_events = fixed_evs
            print(f'  ✓ 字幕文字数超過 → Claude修正ループ完了')
        except Exception as _ae:
            print(f'  ⚠ Claude字幕修正失敗（強制トリム）: {_ae}')
            sub_events = [
                re.sub(r'(.*pos\(\d+,\d+\)\\fs\d+\})(.{13,})$',
                       lambda m2: m2.group(1) + m2.group(2)[:12], ev)
                for ev in sub_events
            ]

    return sub_events, len(violations) == 0
def generate_bgm(duration_sec, sr=44100):
    """128BPM アップテンポBGM（ピーク正規化済み・amixで音量制御）"""
    n = int(sr * duration_sec)
    bgm = np.zeros(n, dtype=np.float32)
    bpm = 128; beat = 60.0 / bpm
    scale = [261.63, 293.66, 329.63, 392.00, 440.00, 523.25, 587.33, 659.25]
    pattern = [0, 2, 4, 5, 4, 2, 0, 2, 4, 7, 6, 5, 4, 2, 0, 4]
    for i in range(int(duration_sec / beat) + len(pattern) + 1):
        pi = i % len(pattern); start = int(i * beat * sr)
        if start >= n: break
        end = min(int((i * beat + beat * 0.75) * sr), n); sz = end - start
        if sz <= 0: continue
        nt = np.linspace(0, sz/sr, sz); freq = scale[pattern[pi] % len(scale)]
        wav = (np.sin(2*np.pi*freq*nt)*0.5 + np.sin(2*np.pi*freq*2*nt)*0.15 + np.sin(2*np.pi*freq*3*nt)*0.08)
        env = np.ones(sz); a, r2 = min(int(0.01*sr), sz), min(int(0.06*sr), sz)
        env[:a] = np.linspace(0, 1, a); env[-r2:] *= np.linspace(1, 0, r2)
        bgm[start:end] += wav * env * 0.22
    for bi in range(int(duration_sec / beat) + 1):
        p = int(bi * beat * sr)
        if p >= n: break
        kl = min(int(0.07*sr), n-p)
        if kl <= 0: continue
        kt = np.linspace(0, 0.07, kl)
        bgm[p:p+kl] += (np.sin(2*np.pi*(90-60*kt/0.07)*kt) * np.exp(-kt*22) * (0.55 if bi%2==0 else 0.22))
    hh_step = beat / 2; rng = np.random.default_rng(42)
    for hi in range(int(duration_sec / hh_step) + 1):
        hp = int(hi * hh_step * sr)
        if hp >= n: break
        hl = min(int(0.018*sr), n-hp)
        if hl <= 0: continue
        ht = np.linspace(0, 0.018, hl)
        bgm[hp:hp+hl] += rng.standard_normal(hl) * np.exp(-ht*90) * 0.07
    bass = [130.81, 130.81, 196.00, 146.83]; bar = beat * 4
    for bi in range(int(duration_sec / bar) + 2):
        p = int(bi * bar * sr)
        if p >= n: break
        bl = min(int(bar*0.88*sr), n-p)
        if bl <= 0: continue
        bt = np.linspace(0, bl/sr, bl)
        bgm[p:p+bl] += np.sin(2*np.pi*bass[bi%len(bass)]*bt) * np.exp(-bt*0.8) * 0.38
    mx = np.max(np.abs(bgm))
    if mx > 0: bgm = bgm / mx
    return bgm

def make_cover_crop_vf(iw2, ih2):
    """入力サイズ → 9:16カバークロップ用 ffmpegフィルタ文字列。黒帯ゼロ保証。"""
    # COVER: AR保持のまま両辺がW/H以上になる最小スケール（コンマなし・パース安全）
    if iw2 * H > ih2 * W:       # 横長素材 → 高さ基準でスケール
        sw = iw2 * H // ih2; sh = H
    else:                        # 縦長素材 → 幅基準でスケール
        sw = W; sh = ih2 * W // iw2
    cx2 = (sw - W) // 2; cy2 = (sh - H) // 2  # 中心基準クロップ座標
    return f'scale={sw}:{sh},crop={W}:{H}:{cx2}:{cy2}'

def make_clip(img_path, dur, out, idx=0, boost=1.0, static_zoom=None):
    """静止画クリップ: 動的Ken Burns or 静的センターズーム。
    static_zoom: float指定時(例:1.25)は静止画を1/zoom倍クロップ→スケールアップ（ジャンプカット後半用）
    """
    d = max(dur, 0.1)
    N = f"{d * FPS:.1f}"
    ZS = f"{1.05 * boost:.4f}"  # zoom-in 開始倍率
    ZE = f"{1.25 * boost:.4f}"  # zoom-in 終了倍率 / zoom-out 開始倍率
    ZD = f"{0.20 * boost:.4f}"  # ズーム変化幅
    # n=出力フレーム番号。コンマ不使用でffmpegフィルタチェーンのパースバグを回避
    styles = [
        # 0: zoom-in(1.05→1.25) + 右パン
        (f"iw/({ZS}+{ZD}*n/{N})", f"ih/({ZS}+{ZD}*n/{N})",
         f"(iw-iw/({ZS}+{ZD}*n/{N}))*n/{N}", f"(ih-ih/({ZS}+{ZD}*n/{N}))/2"),
        # 1: zoom-in(1.05→1.25) + 下パン
        (f"iw/({ZS}+{ZD}*n/{N})", f"ih/({ZS}+{ZD}*n/{N})",
         f"(iw-iw/({ZS}+{ZD}*n/{N}))/2", f"(ih-ih/({ZS}+{ZD}*n/{N}))*n/{N}"),
        # 2: zoom-out(1.25→1.05) + 左パン
        (f"iw/({ZE}-{ZD}*n/{N})", f"ih/({ZE}-{ZD}*n/{N})",
         f"(iw-iw/({ZE}-{ZD}*n/{N}))*(1-n/{N})", f"(ih-ih/({ZE}-{ZD}*n/{N}))/2"),
        # 3: zoom-out(1.25→1.05) + 上パン
        (f"iw/({ZE}-{ZD}*n/{N})", f"ih/({ZE}-{ZD}*n/{N})",
         f"(iw-iw/({ZE}-{ZD}*n/{N}))/2", f"(ih-ih/({ZE}-{ZD}*n/{N}))*(1-n/{N})"),
    ]
    cw, ch, cx, cy = styles[idx % len(styles)]
    if static_zoom and Path(img_path).exists():
        # 静的センターズーム: 1/static_zoom クロップ → W x H スケールアップ
        try:
            _pb2 = subprocess.run(['ffprobe','-v','quiet','-print_format','json','-show_streams',str(img_path)],
                                  capture_output=True, text=True, timeout=10)
            _vs2 = next((s for s in _json.loads(_pb2.stdout).get('streams',[])
                         if s.get('codec_type')=='video'), None)
            _iw2, _ih2 = (int(_vs2['width']), int(_vs2['height'])) if _vs2 else (W, H)
        except Exception:
            _iw2, _ih2 = W, H
        cover2 = make_cover_crop_vf(_iw2, _ih2)
        _zw = int(W / static_zoom); _zh = int(H / static_zoom)
        _zx = (W - _zw) // 2;       _zy = (H - _zh) // 2
        vf_sz = f"{cover2},scale={W}:{H},crop={_zw}:{_zh}:{_zx}:{_zy},scale={W}:{H},{_CINEMA_VF}"
        _enc2 = _get_encoder()
        if _enc2 == 'h264_nvenc':
            ff('-loop','1','-i',str(img_path),'-vf',vf_sz,
               '-t',str(dur),'-r',str(FPS),'-an',
               '-c:v','h264_nvenc','-preset','p4','-cq','20','-pix_fmt','yuv420p',str(out))
        else:
            ff('-loop','1','-i',str(img_path),'-vf',vf_sz,
               '-t',str(dur),'-r',str(FPS),'-an',
               '-c:v','libx264','-preset','fast','-crf','23','-threads','0','-pix_fmt','yuv420p',str(out))
        return
    if Path(img_path).exists():
        # 9:16カバークロップ: 画像の実寸をプローブして黒帯ゼロを保証
        try:
            _pb = subprocess.run(['ffprobe','-v','quiet','-print_format','json','-show_streams',str(img_path)],
                                 capture_output=True, text=True, timeout=10)
            _vs = next((s for s in _json.loads(_pb.stdout).get('streams',[])
                        if s.get('codec_type')=='video'), None)
            _iw, _ih = (int(_vs['width']), int(_vs['height'])) if _vs else (W, H)
        except:
            _iw, _ih = W, H
        cover = make_cover_crop_vf(_iw, _ih)
        # センタークロップ → Ken Burns → 映像覚醒フィルター（1コマンドチェーン）
        vf = f"{cover},crop={cw}:{ch}:{cx}:{cy},scale={W}:{H},{_CINEMA_VF}"
        _enc = _get_encoder()
        if _enc == 'h264_nvenc':
            ff('-loop','1','-i',str(img_path),'-vf',vf,
               '-t',str(dur),'-r',str(FPS),'-an',
               '-c:v','h264_nvenc','-preset','p4','-cq','20',
               '-pix_fmt','yuv420p',str(out))
        else:
            ff('-loop','1','-i',str(img_path),'-vf',vf,
               '-t',str(dur),'-r',str(FPS),'-an',
               '-c:v','libx264','-preset','fast','-crf','23',
               '-threads','0','-pix_fmt','yuv420p',str(out))
    else:
        ff('-f','lavfi','-i',f'color=c=black:s={W}x{H}:r={FPS}',
           '-t',str(dur),'-vf',_CINEMA_VF,
           '-c:v','libx264','-preset','ultrafast','-threads','0',str(out))

def make_gradient_bg(out_path, scene_idx=0, text=''):
    """Pexels取得失敗時の代替: シーンごとに色が変わるグラデーション背景+テキスト"""
    # BGR: 6種類のカラーテーマ（scene_idxで循環）
    _themes = [
        ([77,  0, 38], [80,  0,  0]),   # パープル→ダークレッド
        ([0,  60, 20], [0,  80, 40]),   # ダークグリーン→エメラルド
        ([80, 40,  0], [60, 20,  0]),   # ダークオレンジ→ブラウン
        ([0,  40, 80], [0,  20, 60]),   # ディープブルー→ネイビー
        ([60,  0, 60], [30,  0, 80]),   # ダークマゼンタ→インディゴ
        ([0,  60, 60], [0,  40, 40]),   # ダークティール→シアン
    ]
    c1_l, c2_l = _themes[scene_idx % len(_themes)]
    c1 = np.array(c1_l, dtype=np.float32)
    c2 = np.array(c2_l, dtype=np.float32)
    img = np.zeros((H, W, 3), dtype=np.uint8)
    for y in range(H):
        t = y / H
        img[y, :] = (c1 + (c2 - c1) * t).astype(np.uint8)
    cv2.imwrite(str(out_path), img)
    if text:
        try:
            from PIL import Image as _PILImage, ImageDraw as _Draw, ImageFont as _Font
            pil_img = _PILImage.open(str(out_path)).convert('RGB')
            draw = _Draw.Draw(pil_img)
            _font_path = '/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc'
            try:
                font = _Font.truetype(_font_path, 72)
                font_sm = _Font.truetype(_font_path, 48)
            except Exception:
                font = _Font.load_default()
                font_sm = font
            # シーン番号
            draw.text((60, 80), f'Scene {scene_idx + 1}', font=font_sm, fill=(200, 200, 200))
            # セリフテキスト（折り返し）
            _max_w = W - 120
            _chars_per_line = max(10, _max_w // 72)
            _lines = []
            _remaining = text
            while _remaining:
                _lines.append(_remaining[:_chars_per_line])
                _remaining = _remaining[_chars_per_line:]
            _y = H // 2 - len(_lines) * 80
            for _line in _lines:
                bbox = draw.textbbox((0, 0), _line, font=font)
                _lw = bbox[2] - bbox[0]
                _x = (W - _lw) // 2
                draw.text((_x + 3, _y + 3), _line, font=font, fill=(0, 0, 0))
                draw.text((_x, _y), _line, font=font, fill=(255, 255, 255))
                _y += 90
            pil_img.save(str(out_path))
        except Exception as _pe:
            pass

def fetch_pexels_image_robust(kw, img_fn, scene_idx=0, scene_text=''):
    """1KW 1回検索: portrait固定・401は10秒待ちリトライ・グラデBGは最終手段"""
    _pkey = ''.join(c for c in (PEXELS_API_KEY or '') if 0x21 <= ord(c) <= 0x7E)
    if not _pkey:
        print('  ⚠ PEXELS_API_KEY未設定 → グラデBG')
        make_gradient_bg(img_fn, scene_idx, scene_text)
        return

    def _has_jp(s):
        return bool(re.search(r'[\u3040-\u309F\u30A0-\u30FF\u4E00-\u9FFF]', s))

    def _try_fetch(query):
        """1キーワードで写真取得を試みる。成功したらTrueを返す"""
        _pexels_wait()
        try:
            params = {'query': query, 'per_page': 10, 'orientation': 'portrait'}
            r = _req.get('https://api.pexels.com/v1/search',
                         headers={'Authorization': _pkey},
                         params=params, timeout=15)
            print(f'    [img] {query!r} HTTP={r.status_code}', end='')
            if r.status_code == 401:
                print(' ← レート制限? 10秒待機...', end='')
                time.sleep(10)
                _PEXELS_LAST_T[0] = 0.0  # 次の_pexels_waitをすぐ通過
                _pexels_wait()
                r = _req.get('https://api.pexels.com/v1/search',
                             headers={'Authorization': _pkey},
                             params=params, timeout=15)
                print(f' retry HTTP={r.status_code}', end='')
            if r.status_code == 429:
                print(' ← レート制限 30秒待機...', end='')
                time.sleep(30)
                _PEXELS_LAST_T[0] = 0.0
                return False
            if r.status_code != 200:
                print(' ← スキップ')
                return False
            photos = r.json().get('photos', [])
            if not photos:
                print(' ← 0件')
                return False
            url = (photos[0]['src'].get('portrait')
                   or photos[0]['src'].get('large2x')
                   or photos[0]['src'].get('large'))
            if not url:
                print(' ← URL取得失敗')
                return False
            dl = _req.get(url, timeout=30)
            if dl.status_code == 200 and len(dl.content) > 2048:
                arr = np.frombuffer(dl.content, np.uint8)
                img_check = cv2.imdecode(arr, cv2.IMREAD_COLOR)
                if img_check is not None:
                    img_fn.write_bytes(dl.content)
                    print(f' ✓ ({len(dl.content)//1024}KB)')
                    return True
            print(f' ← DL失敗')
            return False
        except Exception as _e:
            print(f' ← 例外: {_e}')
            return False

    kw_clean = kw.strip()
    # キーワード候補リスト（最大3つ: 元KW → 先頭1語 → 汎用）
    _candidates = []
    if not _has_jp(kw_clean):
        _candidates.append(kw_clean)
        _fw = kw_clean.split()[0]
        if _fw != kw_clean and not _has_jp(_fw):
            _candidates.append(_fw)
    _candidates.append('lifestyle healthy')  # 汎用フォールバック

    for _q in _candidates:
        if _try_fetch(_q):
            return
    print(f'  ⚠ 全キーワード失敗 → グラデーション背景')
    make_gradient_bg(img_fn, scene_idx, scene_text)

def fetch_pexels_video(kw):
    """Pexelsから縦向き12秒以内の動画URLを取得（診断ログ付き）"""
    _pkey = ''.join(c for c in (PEXELS_API_KEY or '') if 0x21 <= ord(c) <= 0x7E)
    if not _pkey:
        return None, 0
    try:
        import random as _rnd2
        # orientation省略（縦動画は少ない → 横動画もffmpegでクロップ変換）
        # per_page=15・duration<=30 で候補を最大化
        for _orient in [None, 'portrait']:  # まず制限なし、次に縦限定で2回試行
            _pexels_wait()
            _params = {'query': kw, 'per_page': 15}
            if _orient: _params['orientation'] = _orient
            r = _req.get('https://api.pexels.com/videos/search',
                         headers={'Authorization': _pkey},
                         params=_params, timeout=15)
            print(f'    [vid] {kw!r} orient={_orient} HTTP={r.status_code}', end='')
            if r.status_code == 401:
                print(' ← APIキー無効')
                return None, 0
            if r.status_code == 429:
                print(' ← レート制限')
                return None, 0
            if r.status_code != 200:
                print(f' ← スキップ')
                continue
            videos = r.json().get('videos', [])
            print(f' {len(videos)}件')
            _eligible = []
            for v in videos:
                vdur = v.get('duration', 999)
                if vdur > 30: continue  # 30秒以内（ffmpegで必要分だけ使用）
                files = sorted(v.get('video_files', []),
                               key=lambda x: x.get('height', 0), reverse=True)
                for f in files:
                    if f.get('height', 0) >= 480:
                        _eligible.append((f['link'], vdur))
                        break
            if _eligible:
                return _rnd2.choice(_eligible)
    except Exception as _e:
        print(f' ← 例外: {_e}')
    return None, 0

def make_clip_from_video(video_path, dur, out, idx=0, boost=1.0):
    """Pexels動画を縦型1080x1920にクロップ・ループ変換（動的 Ken Burns 付き）"""
    probe = subprocess.run(['ffprobe','-v','quiet','-print_format','json','-show_streams',str(video_path)],
                           capture_output=True, text=True)
    try:
        vs = next((s for s in _json.loads(probe.stdout).get('streams',[]) if s.get('codec_type')=='video'), None)
        vw, vh = (int(vs['width']), int(vs['height'])) if vs else (W, H)
    except:
        vw, vh = W, H
    cover = make_cover_crop_vf(vw, vh)  # 9:16カバークロップ（黒帯ゼロ・中心基準）
    d = max(dur, 0.1)
    N = f"{d * FPS:.1f}"
    ZS = f"{1.05 * boost:.4f}"  # zoom-in 開始倍率
    ZE = f"{1.25 * boost:.4f}"  # zoom-in 終了倍率 / zoom-out 開始倍率
    ZD = f"{0.20 * boost:.4f}"  # ズーム変化幅
    # n=出力フレーム番号。コンマ不使用でffmpegフィルタチェーンのパースバグを回避
    kb = [
        # zoom-in(1.05→1.25) + 右パン
        (f"iw/({ZS}+{ZD}*n/{N})", f"ih/({ZS}+{ZD}*n/{N})",
         f"(iw-iw/({ZS}+{ZD}*n/{N}))*n/{N}", f"(ih-ih/({ZS}+{ZD}*n/{N}))/2"),
        # zoom-out(1.25→1.05) + 左パン
        (f"iw/({ZE}-{ZD}*n/{N})", f"ih/({ZE}-{ZD}*n/{N})",
         f"(iw-iw/({ZE}-{ZD}*n/{N}))*(1-n/{N})", f"(ih-ih/({ZE}-{ZD}*n/{N}))/2"),
    ]
    kcw, kch, kcx, kcy = kb[idx % len(kb)]
    ff('-stream_loop','-1','-i',str(video_path),
       '-vf',f'{cover},crop={kcw}:{kch}:{kcx}:{kcy},scale={W}:{H}',
       '-t',str(dur),'-r',str(FPS),'-an',
       '-c:v','libx264','-preset','fast','-crf','23','-pix_fmt','yuv420p',str(out))


# ── ジャンルブラックリスト（テーマ無関係素材の混入を100%防ぐ）────────
_KEYWORD_BLACKLIST = {
    # IT / Tech / Data
    'data','graph','chart','spreadsheet','code','server','network','laptop',
    'computer','keyboard','monitor','dashboard','analytics','database',
    'algorithm','software','programming','cloud','cyber','digital screen',
    # Finance / Business / Office
    'revenue','profit','stock','market','investment','finance','meeting',
    'office','boardroom','presentation','report','business','corporate',
    'dollar','money chart','financial','budget','sales','workspace','desk',
    # Generic / Low-visual-quality
    'lifestyle','person','woman','man','people','human',
}

def sanitize_visual_keyword(kw: str, theme: str) -> str:
    """ブラックリスト語を含むキーワードをテーマ合致の安全なものへ置換"""
    kw_lower = kw.lower()
    for bad in _KEYWORD_BLACKLIST:
        if bad in kw_lower:
            try:
                safe = call_ai(
                    f'テーマ「{theme}」の動画シーンに合う英語の映像キーワードを1〜3語で答えてください。\n'
                    f'例: healthy food / morning routine / fresh vegetables\n'
                    f'キーワードのみ出力（説明不要）',
                    tokens=30
                ).strip().lower()
                if any(b in safe for b in _KEYWORD_BLACKLIST):
                    safe = 'healthy food'
                print(f'    🚫 BL置換: {kw!r} → {safe!r}')
                return safe
            except Exception:
                pass
            return 'healthy food'
    return kw
W, H, FPS = 1080, 1920, 30

# ── 映像覚醒フィルター: シネマグレーディング + アンシャープマスク ────
# unsharp: 輪郭強調（シズル感・湯気・ディテール）
# eq: 彩度1.25倍 + コントラスト微増
# curves: シャドウに青みを足してシネマティックに / ハイライトはクリア
_CINEMA_VF = (
    'unsharp=luma_msize_x=5:luma_msize_y=5:luma_amount=0.8'
    ':chroma_msize_x=3:chroma_msize_y=3:chroma_amount=0.4,'
    'eq=saturation=1.25:contrast=1.03,'
    "curves="
    "r='0/0 0.08/0.06 0.5/0.5 0.9/0.95 1/1':"
    "g='0/0 0.08/0.07 0.5/0.5 0.9/0.93 1/1':"
    "b='0/0.04 0.12/0.15 0.5/0.52 0.9/0.93 1/0.98'"
)

# ── ハードウェアエンコーダ検出（nvenc → libx264 フォールバック） ────
_HW_ENC = [None]  # キャッシュ: None=未検出
def _get_encoder():
    if _HW_ENC[0] is not None:
        return _HW_ENC[0]
    try:
        _t = subprocess.run(
            ['ffmpeg','-hide_banner','-f','lavfi','-i','color=black:s=64x64:r=1',
             '-t','0.1','-c:v','h264_nvenc','-f','null','-'],
            capture_output=True, timeout=5)
        if _t.returncode == 0:
            _HW_ENC[0] = 'h264_nvenc'
            print('  \U0001f680 GPU: h264_nvenc 検出')
            return _HW_ENC[0]
    except Exception:
        pass
    _HW_ENC[0] = 'libx264'
    return _HW_ENC[0]

# ── Pexels グローバルレートリミッター（全APIコールを4秒間隔に制限）───
_PEXELS_LAST_T = [0.0]
def _pexels_wait():
    gap = 4.0 - (time.time() - _PEXELS_LAST_T[0])  # 4s間隔でレート制限回避
    if gap > 0:
        time.sleep(gap)
    _PEXELS_LAST_T[0] = time.time()

# VOICEVOX定数（セル3で設定済みの値を継承、未実行時はデフォルト）
try: VV_AVAILABLE
except NameError: VV_AVAILABLE = False
try: VV_URL
except NameError: VV_URL = 'http://127.0.0.1:50021'
try: VV_SPEAKER
except NameError: VV_SPEAKER = 3
VIDEO_LEAD = 0.05  # シーン2以降: 映像カットが音声より50ms先行
# 全シーン: 動画API優先で取得（VIDEO_SCENE_IDX廃止）
OUTPUT_DIR = Path('/content/output')
OUTPUT_DIR.mkdir(exist_ok=True)
completed = []
failed = []

def generate_visual_keywords(scene_texts, theme):
    """Step4: 確定したセリフからシーン固有の英語視覚キーワードを生成"""
    scenes_str = '\n'.join(f'シーン{i+1}: {t}' for i, t in enumerate(scene_texts))
    try:
        result = call_ai(
            f'各シーンのセリフを読んで、そのシーンの映像として最適な英語キーワードを生成してください。\n\n'
            f'【テーマ】{theme}\n'
            f'【セリフ一覧】\n{scenes_str}\n\n'
            f'【ルール】\n'
            f'① 各シーンで必ず違うキーワード（重複禁止）\n'
            f'② セリフの内容を映像化できる英語2〜3語の名詞フレーズ\n'
            f'③ NG: lifestyle / person / woman / man / healthy（単体）\n'
            f'④ OK: protein shake blender / steamed broccoli / morning alarm clock / '
            f'gym weights / fresh salad / glass of water / '
            f'running shoes / bedroom night / meal prep / coffee mug\n\n'
            f'【出力（英語のみ・厳守）】\n'
            f'scene1: （英語キーワード）\n'
            f'（全{len(scene_texts)}シーン分出力）',
            tokens=300
        )
        kws = []
        for line in result.strip().split('\n'):
            m = re.match(r'scene\d+[:\s]+(.+)', line.strip(), re.I)
            if m:
                kw = m.group(1).strip()
                if re.search(r'[\u3040-\u309F\u30A0-\u30FF\u4E00-\u9FFF]', kw):
                    kw = 'food close up'
                kws.append(kw)
        if len(kws) >= len(scene_texts):
            kws = [sanitize_visual_keyword(k, theme) for k in kws]
            print(f'  ✓ 視覚KW: {kws}')
            return kws[:len(scene_texts)]
    except Exception as _e:
        print(f'  ⚠ 視覚KW生成失敗: {_e}')
    return None

def rewrite_script_to_natural(scene_texts, theme, angle):
    """Step2: AI生成台本を日本人女性インフルエンサーの自然口語に完全リライト"""
    if not scene_texts:
        return scene_texts
    scenes_str = '\n'.join(f'シーン{i+1}: {t}' for i, t in enumerate(scene_texts))
    try:
        rewritten = call_ai(
            f'あなたは日本の人気女性YouTuberです。以下の台本を、スマホカメラに向かって'
            f'実際に話しているような100%自然な口語にリライトしてください。\n\n'
            f'【元の台本】\n{scenes_str}\n\n'
            f'【テーマ】{theme}  【タイトル】{angle}\n\n'
            f'【絶対に守るルール】\n'
            f'・「〜なんだ」「〜なの？」「〜するんだよ」「〜んです」など\n'
            f'  ロボット調・直訳調の語尾は完全に禁止\n'
            f'・語尾は「〜だよ！」「〜よね！」「〜してみて！」「〜のがポイント！」\n'
            f'  「〜じゃない？」など女性インフルエンサーが実際に使う語尾に統一\n'
            f'・①②③番号付きtipsは「〜する！」「〜を選ぶ！」の言い切り型に変換\n'
            f'・シーン1は「え、マジ？」「これ知ってた？」「実はね、」のような\n'
            f'  視聴者が思わず手を止める自然なフックで始める\n'
            f'・各セリフは10〜14文字の短いひとこと（長い文は絶対NG）\n'
            f'・ひらがな・カタカナ・漢字のみ（英語・記号・ハッシュタグ完全禁止）\n'
            f'・【コメント誘発】全シーンの中に1〜2箇所、視聴者が思わず突っ込むフレーズを入れる\n'
            f'  （例：「〜は絶対やるな！」「知らないと一生損する」「〜したら人生変わった」）\n\n'
            f'【出力フォーマット（厳守）】\n'
            f'セリフのみを出力。説明・コメント・空行は一切不要。\n'
            f'シーン1: （リライト後のセリフのみ）\n'
            f'シーン2: （リライト後のセリフのみ）\n'
            f'（元と同じシーン数で出力すること）',
            tokens=1024
        )
        # リライト結果をシーン単位でパース
        result = []
        for line in rewritten.strip().split('\n'):
            m = re.match(r'シーン\d+[:：]\s*(.+)', line.strip())
            if m:
                t = clean_text(m.group(1).strip())
                if t: result.append(t)
        # シーン数が一致しない場合は元テキストで補完
        while len(result) < len(scene_texts):
            result.append(scene_texts[len(result)])
        return result[:len(scene_texts)]
    except Exception as e:
        print(f'  ⚠ リライト失敗（元テキストにフォールバック）: {e}')
        return scene_texts

def make_one_video(idx, angle):
    vid_num = idx + 1
    safe_angle = re.sub(r'[\\/:*?"<>|\s]','_',angle)[:25]
    print(f'\n{"─"*50}')
    print(f'🎬 [{vid_num}/{VIDEO_COUNT}] {angle}')
    print(f'{"─"*50}')
    # ── チェックポイント: 既に完成済みならスキップ ─────────
    safe_theme_chk = re.sub(r'[\\/:*?"<>|\s]','_',THEME)[:15]
    safe_angle_chk = re.sub(r'[\\/:*?"<>|\s]','_',angle)[:25]
    OUT_CHK = OUTPUT_DIR/f'{vid_num:02d}_{safe_theme_chk}_{safe_angle_chk}.mp4'
    if OUT_CHK.exists() and OUT_CHK.stat().st_size > 50000:
        print(f'  ⏭️  スキップ（完成済み: {OUT_CHK.name}）')
        return str(OUT_CHK)

    TMP = Path(tempfile.mkdtemp(prefix=f'yt{vid_num}_'))
    MEDIA = TMP/'media'; MEDIA.mkdir()

    # ── 台本生成（シーン数はClaudeが内容に応じて自動決定） ────
    print('  📝 台本生成中...')
    _trend_ctx = globals().get('TREND_ANALYSIS', '')[:600]
    _trend_block = (
        f'【最新トレンド分析（実際のYouTube上位動画から抽出）】\n{_trend_ctx}\n\n'
        if _trend_ctx else ''
    )
    raw = call_ai(
        f'{_trend_block}'
        f'YouTube Shorts台本を作成してください。\n'
        f'【タイトル】{angle}\n【テーマ】{THEME}\n'
        f'【目標秒数】{DURATION}秒前後（15〜60秒の範囲）\n\n'
        f'【シーン数の決め方（重要）】\n'
        f'・シンプルな1ポイントtips → 3〜4シーン\n'
        f'・方法・コツ系 → 5〜6シーン\n'
        f'・詳しい解説系 → 7〜8シーン\n'
        f'内容の複雑さに合わせて自由にシーン数を決めてください\n\n'
        f'【絶対ルール】\n'
        f'・ハッシュタグ（#）・記号（@&*[]等）・英単語・URL完全禁止\n'
        f'・ひらがな・カタカナ・漢字のみ\n'
        f'・各セリフは10〜18文字の短いひとこと\n'
        f'・感情の起伏（驚き→共感→解決→期待）\n'
        f'・語尾: 〜だよ！/〜してみて/〜のがコツ/〜が大事/〜だよね/〜してみよう\n'
        f'・「〜なの？」「〜むんだよ」など直訳風・不自然な語尾は絶対NG\n'
        f'・読点（、）で自然な間（例：実は、知ってた？/毎朝、やってみて！）\n'
        f'・シーン1は衝撃フック（これ知ってた？/知らないと損！/えっ本当に？ 等）\n'
        f'・①②③の番号付きtipsは必ず言い切り型「〜する！」（例:①白湯を朝に飲む！②食前に一杯！）\n'
        f'・【コメント誘発ルール】全シーンの中に1〜2箇所、視聴者がコメント欄で突っ込みたくなる\n'
        f'  「極端な断言・煽り」を入れる（例:「〜は絶対やるな！」「知らないと一生損する」\n'
        f'  「これやめたら人生変わった」「〜してる人は今すぐやめて」）\n\n'
        f'【キーワードルール（重要）】\n'
        f'・各シーンで映すべき具体的な映像を英語1〜2語で指定する\n'
        f'・NG例:「lifestyle」「woman」「person」\n'
        f'・OK例:「healthy food」「warm water drink」「grilled chicken」「bedroom clock」\n'
        f'・シーン1（フック）は必ず料理・食材・具体的な行動が映るキーワード\n\n'
        f'[台本]\n'
        f'シーン1（フック）: \n'
        f'シーン2（共感）: \n'
        f'シーン3（①1つ目 or 解説1）: \n'
        f'（以降、内容に応じたシーン数まで続ける）\n\n'
        f'[キーワード]\n'
        f'scene1: （料理・食材など具体的な英語1〜2語）\n'
        f'（台本と同じシーン数）'
    )

    script_text = (re.search(r'\[台本\]([\s\S]*?)(?=\[キーワード\])', raw) or
                   type('x',(),{'group':lambda s,n:raw})()).group(1)
    kw_block = re.search(r'\[キーワード\]([\s\S]*?)$', raw)

    # 実際に生成されたシーン数をカウント（動的SCENE_COUNT）
    scene_lines = re.split(r'シーン\d+[（(][^)）]*[)）]\s*[:：]', script_text)
    scene_texts = [clean_text(s) for s in scene_lines[1:] if s.strip()]
    SCENE_COUNT = max(3, min(8, len(scene_texts)))
    while len(scene_texts) < SCENE_COUNT: scene_texts.append(THEME)
    scene_texts = scene_texts[:SCENE_COUNT]

    keywords = []
    if kw_block:
        for line in kw_block.group(1).split('\n'):
            m = re.match(r'scene\d+[:\s]+(.+)', line.strip(), re.I)
            if m: keywords.append(m.group(1).strip())
    while len(keywords) < SCENE_COUNT: keywords.append('lifestyle')
    keywords = keywords[:SCENE_COUNT]
    print(f'  ✓ Step1 台本生成完了（{SCENE_COUNT}シーン）')

    # ── Step 2: 日本語校正 & SNS自然口語リライト ──────────────
    print('  ✏️  Step2: 自然口語リライト中（インフルエンサー調）...')
    scene_texts = rewrite_script_to_natural(scene_texts, THEME, angle)
    if scene_texts:
        print(f'  ✓ リライト完了 scene1→ {scene_texts[0]}')

    # ── Step 3: 文法・内容・フォーマット自己検証 & 自動修正 ─────
    print('  🔍 Step3: 文法・内容を自己検証 & 自動修正中...')
    scene_texts = verify_and_fix_script(scene_texts, THEME, angle)
    if scene_texts:
        print(f'  ✓ 検証完了 scene1→ {scene_texts[0]}')

    # ── ループ構造対応 ───────────────────────────────────────────
    try: _loop_mode = LOOP_STRUCTURE
    except NameError: _loop_mode = True
    if _loop_mode and SCENE_COUNT >= 3:
        import random as _rnd
        _loop_ctas = [
            'もう一度最初から確認してみて！',
            'これ保存して何度も見てみて！',
            'コメントで「やる」って教えて！',
            'もう一回再生して確かめてみて！',
            '最初から見てたか確認してみて！',
        ]
        scene_texts[-1] = _rnd.choice(_loop_ctas)
        keywords[-1]    = keywords[0]  # 最後KW=最初KW（視覚的ブックエンド）
        print(f'  🔄 ループ構造: 最終シーン → 「{scene_texts[-1]}」/ KW={keywords[0]!r}')
    # ── Step 4: セリフから視覚キーワードを再生成（シーン固有・重複なし）──
    print('  🔍 Step4: セリフから視覚キーワードを再生成中...')
    _new_kw = generate_visual_keywords(scene_texts, THEME)
    if _new_kw:
        keywords = _new_kw
        # ループ構造でkeywords[-1]を上書き済みの場合も反映
        if _loop_mode and SCENE_COUNT >= 3:
            keywords[-1] = keywords[0]
    else:
        print('  ⚠ 再生成失敗 → 元のキーワードを使用')

    # TTS用テキスト（語尾・間補正済み）を字幕テキストと分離して生成
    tts_texts = [tts_text_cleanse(t) for t in scene_texts]

    # ── メディア取得（動画と静止画を明示的に組み合わせ） ─────
    _pkey_clean = ''.join(c for c in (PEXELS_API_KEY or '') if 0x21 <= ord(c) <= 0x7E)
    _pk = _pkey_clean[:8] + '...' if len(_pkey_clean) > 8 else '(未設定❌)'
    _pkey_ok = bool(_pkey_clean)
    if not _pkey_ok:
        print('  ❌❌ PEXELS_API_KEYが未設定です！')
        print('     → 全シーンにグラデーション背景を使用します')
        print('     → セル1で PEXELS_API_KEY を入力するか、')
        print('        ColabシークレットにPEXELS_API_KEYを登録してください')
    print(f'  🖼 メディア取得中（全シーン動画優先 → 動画なしは静止画フォールバック）')
    print(f'     Pexels key: {_pk} / keywords: {keywords}')
    scenes = []
    for i, kw in enumerate(keywords):
        img_fn = MEDIA/f'img_{i+1:02d}.jpg'
        vid_fn = MEDIA/f'vid_{i+1:02d}.mp4'
        # 写真API のみ使用（動画APIは無料プランでは401になるため無効化）
        fetch_pexels_image_robust(kw, img_fn, i, scene_texts[i] if i < len(scene_texts) else '')
        print(f'    scene{i+1}: 🖼[{kw}] {"✓" if img_fn.exists() else "グラデBG"}')
        scenes.append({'scene':i+1,'image':img_fn,'video':vid_fn,'is_video':False,
                       'keyword':kw,
                       'text': scene_texts[i],
                       'tts':  tts_texts[i]})
    print('  ✓ メディア取得完了（全シーン写真+Ken Burns）')

    # ── ループ構造: 最後のシーンのメディアを最初のシーンのものに完全一致させる ──
    # ── スタイル変換（静止画のみ） ────────────────────────────
    if IMAGE_STYLE in STYLES:
        fn_style = STYLES[IMAGE_STYLE]
        for s in scenes:
            if not s['is_video'] and s['image'].exists():
                img = cv2.imread(str(s['image']))
                if img is not None: cv2.imwrite(str(s['image']), fn_style(img))
        print('  ✓ スタイル変換完了')

    # ── 音声生成（フレーズ分割・句読点ごとに自然な間） ────────
    print('  🎙 音声生成中（フレーズ分割・自然な間）...')
    wavs = []
    for i, s in enumerate(scenes):
        wav_out = make_tts_audio(s.get('tts') or s['text'] or THEME, TMP, f'tts{i}')
        if wav_out and wav_out.exists():
            audio_dur = get_wav_dur(wav_out)
            s['duration'] = max((0.0 if i == 0 else VIDEO_LEAD) + audio_dur, 1.5)
            wavs.append(wav_out)
        else:
            s['duration'] = max(DURATION / SCENE_COUNT, 2.0)
            print(f'  ⚠ 音声スキップ scene{i+1}')

    total_dur = sum(s['duration'] for s in scenes)
    # ── Duration Audit: 音声尺と映像尺を50ms以内に同期 ───────────────
    print('  🔍 Duration Audit: 音声尺を検証中...')
    _wavs_iter = iter(wavs)
    _drift_total = 0.0
    for _ai, _s in enumerate(scenes):
        _wf = next(_wavs_iter, None)
        if _wf and _wf.exists():
            _adur = get_wav_dur(_wf)
            _lead = 0.0 if _ai == 0 else VIDEO_LEAD
            _exp = max(_lead + _adur, 1.5)
            _drift = abs(_s['duration'] - _exp)
            if _drift > 0.05:
                print(f'    ⚠ scene{_ai+1} drift={_drift*1000:.0f}ms → {_s["duration"]:.3f}s→{_exp:.3f}s 修正')
                _s['duration'] = _exp
                _drift_total += _drift
    if _drift_total == 0:
        print('  ✓ Duration Audit: 全シーン同期 OK (±50ms以内)')
    else:
        total_dur = sum(s['duration'] for s in scenes)
        print(f'  ✓ Duration Audit: 修正完了 (総ドリフト{_drift_total*1000:.0f}ms → total={total_dur:.2f}s)')
    cap_dur = min(max(total_dur, 15.0), 25.0)  # YouTube Shorts: 15〜25秒（エンドレスループ最適）
    print(f'  ✓ 音声完了 合計{total_dur:.1f}秒 → 出力{cap_dur:.0f}秒')

    # ── 動画クリップ生成 ──────────────────────────────────────
    print('  🎬 クリップ生成中（3秒超えシーン自動ジャンプカット分割）...')
    SPLIT_THRESHOLD = 2.5  # 2.5秒超えシーンは前半(通常) + 後半(静的1.25xセンターズーム)
    clips = []
    for i_c, s in enumerate(scenes):
        d = s['duration']
        has_vid = s['is_video'] and s['video'].exists()

        def _mk(dur, out_path, kb_idx, boost=1.0, static_zoom=None):
            """ソース種別に応じたクリップ生成（static_zoom=1.25で後半センターズーム）"""
            if has_vid:
                try:
                    make_clip_from_video(s['video'], dur, out_path, idx=kb_idx, boost=boost)
                    return
                except Exception as e:
                    print(f'  ⚠ 動画変換失敗→静止画: {e}')
            make_clip(s['image'], dur, out_path, idx=kb_idx, boost=boost, static_zoom=static_zoom)

        if d > SPLIT_THRESHOLD:
            # 3秒超え: 前半（通常）+ 後半（1.2倍ズームアップ）で疑似ジャンプカット
            half = d / 2
            out1 = TMP/f"c{s['scene']:02d}a.mp4"
            out2 = TMP/f"c{s['scene']:02d}b.mp4"
            _mk(half, out1, i_c, boost=1.0)
            _mk(half, out2, i_c + 1, static_zoom=1.25)
            clips.extend([out1, out2])
            print(f"  ✂ scene{s['scene']} ジャンプカット分割 ({half:.1f}s × 2 / 後半1.25xセンターズーム)")
        else:
            out = TMP/f"c{s['scene']:02d}.mp4"
            _mk(d, out, i_c, boost=1.0)
            clips.append(out)

    lf2 = TMP/'cl.txt'
    lf2.write_text('\n'.join(f"file '{p}'" for p in clips))
    mg = TMP/'m.mp4'
    ff('-f','concat','-safe','0','-i',str(lf2),'-c','copy',str(mg))

    # ── 字幕（黄色・大きめ） ─────────────────────────────────
    ah = (
        f"[Script Info]\nPlayResX:{W}\nPlayResY:{H}\nScriptType:v4.00+\n\n"
        f"[V4+ Styles]\n"
        f"Format:Name,Fontname,Fontsize,PrimaryColour,SecondaryColour,OutlineColour,BackColour,"
        f"Bold,Italic,Underline,StrikeOut,ScaleX,ScaleY,Spacing,Angle,BorderStyle,Outline,Shadow,"
        f"Alignment,MarginL,MarginR,MarginV,Encoding\n"
        f"Style:Default,Noto Sans CJK JP,75,&H0000FFFF,&H000000FF,&H00000000,&H80000000,"
        f"-1,0,0,0,100,100,2,0,3,12,0,2,{int(W*0.10)},{int(W*0.10)},{int(H*0.28)},1\n"
        f"Style:Hook,Noto Sans CJK JP,76,&H00FFFFFF,&H000000FF,&H00000000,&H000000FF,"
        f"-1,0,0,0,100,100,0,0,3,20,0,8,60,60,{int(H*0.05)},1\n\n"
        f"[Events]\nFormat:Layer,Start,End,Style,Name,MarginL,MarginR,MarginV,Effect,Text\n"
    )
    ev = []; t = 0.0
    # ── 冒頭フックテロップ（0〜2秒・Hook スタイル・赤背景）───────────
    if '】' in angle and '【' in angle:
        _hl1 = angle[:angle.index('】')+1]; _hl2 = angle[angle.index('】')+1:].strip()
        _htxt = _hl1 + r'\N' + _hl2 if _hl2 else _hl1
    elif len(angle) > 10:
        _m = len(angle)//2; _htxt = angle[:_m] + r'\N' + angle[_m:]
    else:
        _htxt = angle
    # 行溢れ防止: 最長行が12文字超ならフォントサイズを自動縮小
    # Hook style: 76pt × 1文字≈76px, 有効幅960px (1080-60×2margins) → 最大12文字
    _hook_lines = [l for l in _htxt.split(r'\N') if l]
    _max_hook_len = max(len(l) for l in _hook_lines) if _hook_lines else 1
    if _max_hook_len > 12:
        _hook_fs = max(40, int(960 // _max_hook_len))
        _htxt = r'{\fs' + str(_hook_fs) + r'}' + _htxt
    _hend = min(2.0, scenes[0]['duration'] if scenes else 2.0)
    ev.append(f"Dialogue:0,{at(0)},{at(_hend)},Hook,,0,0,0,,{_htxt}")
    # 字幕: 最大12文字・1行・1〜2秒カード切り替え（Y=62%固定）
    SUB_CX = W // 2
    SUB_CY = int(H * 0.62)  # 60-65%エリア中心（最下部張り付き防止）
    for s in scenes:
        cards = make_sub_cards(s['text'], s['duration'])
        cards = validate_sub_cards(cards, s['text'], s['duration'])
        for card_text, c_s_off, c_e_off in cards:
            c_s = t + c_s_off
            c_e = t + c_e_off
            ev.append(
                f"Dialogue:0,{at(c_s)},{at(max(c_s+0.1, c_e))},Default,,0,0,0,,"
                f"{{\\an5\\pos({SUB_CX},{SUB_CY})\\fs75}}{card_text}"
            )
        t += s['duration']

    sub = TMP/'s.ass'
    sub.write_text(ah+'\n'.join(ev), encoding='utf-8')
    se = str(sub).replace('\\','/').replace(':','\\:')

    # ── ナレーション音声を結合 ────────────────────────────────
    comb = TMP/'vc.wav'; vaac = TMP/'v.aac'
    # シーン1以降の先頭に VIDEO_LEAD 分の無音を挿入（映像50ms先行同期）
    _aparts = []
    for _iw, _wf in enumerate(wavs):
        if _iw > 0:
            _ls = TMP/f'_ls{_iw}.wav'
            wio.write(str(_ls), 44100, np.zeros(int(44100*VIDEO_LEAD), dtype=np.float32))
            _aparts.append(_ls)
        _aparts.append(_wf)
    if _aparts:
        lf = TMP/'vl.txt'
        lf.write_text('\n'.join(f"file '{p}'" for p in _aparts))
        ff('-f','concat','-safe','0','-i',str(lf),'-c','copy',str(comb))
    else:
        wio.write(str(comb), 44100, np.zeros(int(44100*total_dur), dtype=np.float32))
    # ── 無音検知: -40dB 以下が300ms以上続く区間を圧縮 ──────────────────
    print('  🔍 無音検知中 (silencedetect -40dB / 300ms)...')
    _sd = subprocess.run(
        ['ffmpeg','-i',str(comb),'-af','silencedetect=n=-40dB:d=0.3','-f','null','-'],
        capture_output=True, text=True)
    _sd_durs = re.findall(r'silence_duration:\s*([\d.]+)', _sd.stderr)
    if _sd_durs:
        _tot_sil = sum(float(x) for x in _sd_durs)
        print(f'  ⚠ 無音{len(_sd_durs)}箇所 計{_tot_sil:.2f}s → silenceremove適用中...')
        _comb_sr = TMP/'vc_clean.wav'
        try:
            ff('-i',str(comb),'-af',
               'silenceremove=stop_periods=-1:stop_duration=0.3:stop_threshold=-40dB:stop_silence=0.05',
               str(_comb_sr))
            if _comb_sr.exists() and _comb_sr.stat().st_size > 1000:
                comb = _comb_sr
                print(f'  ✓ 無音圧縮完了')
        except Exception as _se:
            print(f'  ⚠ silenceremove失敗（ローフィノイズフロアがカバー）: {_se}')
    else:
        print('  ✓ 無音検知: 問題なし')
    ff('-i',str(comb),'-c:a','aac','-ar','44100',str(vaac))

    # ── BGM生成・ノイズフロア生成・最終合成 ─────────────────────
    bgm_path = TMP/'bgm.wav'
    wio.write(str(bgm_path), 44100, generate_bgm(cap_dur + 2))
    # Lo-Fiノイズフロア: ぶつ切り無音を聴覚的に隠蔽（約-30dB）
    noise_path = TMP/'noise.wav'
    _nsr = 44100; _nn = int(_nsr * (cap_dur + 2))
    _rng = np.random.default_rng(42)
    _raw = _rng.standard_normal(_nn).astype(np.float32)
    # 簡易ローパス（Lo-Fi感: 2kHz以下）: 16サンプル移動平均
    _k = np.ones(16, dtype=np.float32) / 16
    _smooth = np.convolve(_raw, _k, mode='same')
    _smooth = (_smooth / (np.abs(_smooth).max() + 1e-9) * 0.03).astype(np.float32)
    wio.write(str(noise_path), _nsr, _smooth)
    print('  🎵 BGM + ノイズフロア完了')

    # ── シームレスループ: TTS合計尺で0フレーム余白なくカット ─────────
    # total_dur = 各シーンのTTS時間の合計（VIDEO_LEAD 込み）
    # vaac のエンコーダ遅延（≈0.023s）は -t で切り落とす → 末尾無音ゼロ
    cap_dur = round(min(max(total_dur, 15.0), 25.0), 2)
    print(f'  ✂️  シームレスループ: total_dur={total_dur:.3f}s → cap_dur={cap_dur:.2f}s')

    safe_theme_f = re.sub(r'[\\/:*?"<>|\s]','_',THEME)[:15]
    OUT = OUTPUT_DIR/f'{vid_num:02d}_{safe_theme_f}_{safe_angle}.mp4'

    # ── LAYER4: レンダリング前 全仕様自動監査 & 自己修正 ──────────────────
    ev, _audit_ok = pre_render_audit(scenes, ev, cap_dur, keywords, THEME)
    sub.write_text(ah+'\n'.join(ev), encoding='utf-8')  # 修正後ASS再書き込み

    ff('-i',str(mg),'-i',str(vaac),'-i',str(bgm_path),'-i',str(noise_path),
       '-filter_complex',
       '[1:a]volume=1.0[narr];'
       '[2:a]volume=0.65[bgm];'
       '[3:a]volume=1.0[nz];'
       '[narr][bgm][nz]amix=inputs=3:duration=first:normalize=0[aout]',
       '-vf',f'ass={se}',
       '-map','0:v','-map','[aout]',
       '-c:v','libx264','-preset','fast','-crf','20',
       '-c:a','aac','-b:a','192k',
       '-pix_fmt','yuv420p','-movflags','+faststart',
       '-t',str(cap_dur),str(OUT))

    if not OUT.exists() or OUT.stat().st_size < 10000:
        raise RuntimeError(f'動画生成失敗: {OUT}')

    mb = OUT.stat().st_size/1_048_576
    print(f'  ✅ 完成！ {OUT.name} ({mb:.1f}MB, {cap_dur:.0f}秒)')
    return str(OUT)

# ── Pexels APIキー事前確認 ──────────────────────────────────────────────
print('🔑 Pexels APIキー確認中...')
_pk_test = ''.join(c for c in (PEXELS_API_KEY or '') if 0x21 <= ord(c) <= 0x7E)
if not _pk_test:
    print('  ❌ PEXELS_API_KEY が空です！セル1でキーを設定してください。')
else:
    try:
        _tr = _req.get('https://api.pexels.com/v1/search',
                       headers={'Authorization': _pk_test},
                       params={'query': 'food', 'per_page': 1}, timeout=10)
        _rem = _tr.headers.get('X-Ratelimit-Remaining', '不明')
        if _tr.status_code == 200:
            print(f'  ✅ Pexels接続OK  key={_pk_test[:6]}...  残リクエスト={_rem}')
            _PEXELS_LAST_T[0] = time.time()
        elif _tr.status_code == 401:
            print(f'  ❌ Pexels: APIキーが無効 (401) → 全シーンにグラデーション背景')
        elif _tr.status_code == 429:
            print(f'  ⚠ Pexels: レート制限中 (429) → 30秒待機')
            time.sleep(30)
        else:
            print(f'  ⚠ Pexels: HTTP {_tr.status_code}')
    except Exception as _te:
        print(f'  ⚠ Pexels接続テスト失敗: {_te}')
print()

# ── 再開チェック: 既存完成動画を確認 ──────────────────────
import traceback
_safe_th = re.sub(r'[\\/:*?"<>|\s]','_',THEME)[:15]
_already = [f for f in OUTPUT_DIR.glob(f'??_{_safe_th}_*.mp4') if f.stat().st_size > 50000]
if _already:
    print(f'📂 完成済み動画 {len(_already)}本 を検出 → 該当番号はスキップします')
    for _f in sorted(_already): print(f'   ✓ {_f.name}')
else:
    print('📂 完成済みなし → 全本新規生成')

start_all = time.time()
for idx, angle in enumerate(ANGLES):
    try:
        out_path = make_one_video(idx, angle)
        completed.append((idx+1, angle, out_path))
    except Exception as e:
        print(f'\n  ❌ エラー: {traceback.format_exc()[-500:]}')
        failed.append((idx+1, angle, str(e)))

elapsed = (time.time()-start_all)/60
print(f'\n{"="*50}')
print(f'🎉 完了！ 成功:{len(completed)}本 / 失敗:{len(failed)}本 / {elapsed:.1f}分')
print(f'{"="*50}')


In [ ]:
# 【自動】完成動画をダウンロード（触らなくてOK）
from IPython.display import Video, display, Audio
from google.colab import files
import shutil, numpy as np
from pathlib import Path

# completed が未定義・空の場合は /content/output/ から直接スキャン
try:
    _vids = [(n, a, p) for n, a, p in completed if Path(p).exists()]
except NameError:
    _vids = []

if not _vids:
    _found = sorted(Path('/content/output').glob('*.mp4'))
    _vids = [(i+1, p.stem, str(p)) for i, p in enumerate(_found) if p.stat().st_size > 10000]
    if _vids:
        print(f'⚠ completed未定義 → /content/output/ から{len(_vids)}本を検出')

# 結果表示
if not _vids:
    print('❌ ダウンロードできる動画がありません')
    print('   /content/output/ フォルダを左パネルで確認してください')
    try:
        for p in Path('/content/output').iterdir():
            print(f'   {p.name}  {p.stat().st_size//1024}KB')
    except:
        pass
else:
    print(f'✅ 完成 {len(_vids)}本:')
    for num, angle, path in _vids:
        mb = Path(path).stat().st_size / 1_048_576
        print(f'  {num}. [{mb:.1f}MB] {Path(path).name}')

    # プレビュー
    shutil.copy(_vids[0][2], '/content/preview.mp4')
    display(Video('/content/preview.mp4', embed=True, width=360))

    # 通知音
    _sr = 44100
    _b = np.sin(2*np.pi*880*np.linspace(0,.3,int(_sr*.3)))*.5
    display(Audio(np.concatenate([_b,np.zeros(int(_sr*.1)),_b]), rate=_sr, autoplay=True))

    # ダウンロード
    print('\n⬇️ ダウンロード中...')
    for num, angle, path in _vids:
        files.download(path)
        print(f'  ✓ {Path(path).name}')